# Baseline Model: Logistic Regression, Single Feature

In [ ]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("Final_Project_Model1") \
    .getOrCreate()

DATA_FILEPATH = '/standard/ds7200-apt4c/team2/ptbxl_nw'

### Below is the code I used for the supervised learning lab (starting point for the log reg)

#### Packaging variables into a feature column

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["f1", "f2"],
                           outputCol="features")

tr = assembler.transform(train)
te = assembler.transform(test)

tr.select("*").show(2, truncate=False)

#### Standardizing the features

In [ ]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
scalerModel = scaler.fit(tr) 

scaledTrain = scalerModel.transform(tr) 
scaledTest = scalerModel.transform(te)

scaledTrain.select("diagnosis","features","scaledFeatures").show(2, truncate=False)

#### Fitting the model

In [ ]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression

# string labels to numeric labels
indexer = StringIndexer(
    inputCol="diagnosis",
    outputCol="label"
)

indexedTrain = indexer.fit(scaledTrain).transform(scaledTrain)
indexedTest = indexer.fit(scaledTest).transform(scaledTest)

lr = LogisticRegression(labelCol='label',
                        featuresCol='scaledFeatures',
                        maxIter=10, 
                        regParam=0.3, 
                        elasticNetParam=0.8)

lrModel = lr.fit(indexedTrain)

print("Coefficients: " + str(lrModel.coefficients))
print("Intercept: " + str(lrModel.intercept))